# Offline evaluation on `shop.history_sessions`

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [1]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

## Data

In [2]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test
100000 test impressions, observed CTR 0.0374


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased
0,r_74191d5ac418d32b,2026-11-13 06:03:56.359000000,u_22099325,"{'device': 'web', 'page': 'product', 'region':...","[{'product_id': 'p_072126', 'ts': '2026-11-13 ...","[p_1bcbe1, p_703fd0, p_94032a, p_2874e5, p_2cc...","[0.168646, 0.009078, 0.018661, 0.026119, 0.041...",NaN,NaN
1,r_a5f08358f0de4507,2026-11-10 03:54:24.495000000,u_640b24fb,"{'device': 'ios', 'page': 'category', 'region'...","[{'product_id': 'p_437b27', 'ts': '2026-11-10 ...","[p_13be1a, p_c98f7f, p_8936bd, p_473de9, p_ee9...","[0.111716, 0.035069, 0.000702, 0.007621, 0.086...",NaN,NaN
2,r_29b3bbe751491db6,2026-11-12 03:15:57.368000000,u_83fe07a0,"{'device': 'android', 'page': 'home', 'region'...",[],"[p_9ea2ef, p_badc9a, p_dc37f5, p_c43b3c, p_a4e...","[0.001586, 0.011761, 0.00776, 0.002867, 0.0088...",NaN,NaN


In [3]:
import sys
sys.path.insert(0, ".")
from recommender import best_sellers, recommend   # the same code that runs live

best = best_sellers(until=CUT)

In [4]:
# Co-views computed inside RawTree: for every product, the 20 products most viewed in the same sessions
coviews = sql(f"""
    SELECT a.product_id AS p, b.product_id AS q, count() AS n
    FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS a
    INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS b
      ON a.session_id = b.session_id
    WHERE a.product_id != b.product_id
    GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p""")
covisit = coviews.groupby("p").q.apply(list).to_dict()
print(len(covisit), "products with co-views")

5000 products with co-views


## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [5]:
def baseline(row):
    """The live baseline, imported from starter/recommender.py."""
    return recommend(row.recent_views, best)

def coview(row):
    """Products co-viewed with this session's views (most recent first), then the baseline."""
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views)):
        for rank, q in enumerate(covisit.get(v["product_id"], [])):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

## Evaluation

In [6]:
POLICIES = {"baseline": baseline, "coview": coview}

def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0110


/tmp/ipykernel_592697/3756167819.py:11: RuntimeWarning: invalid value encountered in divide
  w = match / df.propensity.str[0].to_numpy()


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.386674,NaN,NaN,3971.0
coview,0.237624,NaN,NaN,5459.0


## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.

___

Métricas:
- hit_rate@10 = (# clicked products in top 10) / (# observed clicks)
- IPC_ctr_slot1
- SNIPS_ctr_slot1
- slot1_matches: # observations where the new policy's first recommendation matches the logging policy's first recommendation

# Two towers v2

**Por qué perdía la v1** (hit@10 0,061 vs 0,390 del baseline):
1. **No sabía qué es popular.** La torre de producto solo veía categoría, subcategoría, marca y precio, así que dos productos de la misma subcategoría y marca eran casi idénticos. El baseline gana precisamente por la popularidad.
2. **Solo negativos del carrusel.** El softmax solo comparaba los 10 productos mostrados, y esos ya los había elegido el sistema antiguo (best-sellers). El modelo aprendía a distinguir entre productos populares, pero nunca veía los 4.990 restantes, que son los que tiene que descartar al rankear el catálogo entero.
3. **Datos pocos y sesgados.** `ORDER BY ts LIMIT 100000` cogía solo el primer día o dos (unos 3.700 positivos) y entrenaba 100 épocas sin validación, así que sobreajustaba.

**Cambios en la v2**
- La torre de producto recibe **popularidad con decaimiento** (vistas, carrito y compras de los últimos 1, 7 y 30 días, calculadas en RawTree) y un **embedding de ID** con *ID-dropout*. El dropout obliga al modelo a funcionar también sin el ID, que es lo que pasa con los productos nuevos (cold start).
- **Negativos mixtos:** los 10 del carrusel (negativos difíciles) más 256 productos aleatorios del catálogo por batch.
- **Datos sin fuga de futuro:** se entrena con los 14 días anteriores a `CUT` y la popularidad se calcula solo con datos anteriores a `TRAIN_START`. Los 2 últimos días se reservan para validar, con early stopping.
- **Mezcla final** `score = two_tower + β · popularidad`, con β elegido en validación (nunca en test). Con β grande se recupera un ranking por popularidad, así que la v2 no debería quedar peor que el baseline.
- Inferencia **en lote** (un único producto matricial): en vivo cuesta milisegundos.

In [7]:
import math, time
import torch, torch.nn as nn, torch.nn.functional as F

torch.manual_seed(0); np.random.seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
MAX_HISTORY = 50          # recent_views rara vez pasa de 50; 500 solo era padding
EMB_DIM = 64
TRAIN_DAYS, VAL_DAYS = 14, 2
TRAIN_START = str((pd.Timestamp(CUT) - pd.Timedelta(days=TRAIN_DAYS)).date())
VAL_START   = str((pd.Timestamp(CUT) - pd.Timedelta(days=VAL_DAYS)).date())
print("pop_train <", TRAIN_START, "<= train <", VAL_START, "<= val <", CUT, "<= test")

pop_train < 2026-10-24 <= train < 2026-11-05 <= val < 2026-11-07 <= test


## Catálogo y popularidad (SQL en RawTree)

Pivotamos por `event_type`, así no tenemos que adivinar los nombres exactos de los eventos. Hay una función `popularity(until)` para que train y test usen cada uno solo su pasado.

In [8]:
catalog = sql("SELECT product_id, brand, category, subcategory, price_eur FROM shop.catalog")
catalog["product_id"] = catalog.product_id.astype(str)
PIDS = catalog.product_id.tolist()
PIDX = {p: i + 1 for i, p in enumerate(PIDS)}        # 0 = desconocido / padding
P = len(PIDS) + 1

def enc(col):
    vals = {v: i + 1 for i, v in enumerate(sorted(catalog[col].astype(str).unique()))}
    return np.r_[0, catalog[col].astype(str).map(vals).to_numpy()], len(vals) + 1
cat_arr, n_cat = enc("category"); sub_arr, n_sub = enc("subcategory"); brand_arr, n_brand = enc("brand")
price_arr = np.r_[0.0, np.log1p(pd.to_numeric(catalog.price_eur, errors="coerce").fillna(0).clip(lower=0))]

def popularity(until):
    df = sql(f"""
        SELECT product_id::String AS product_id, event_type::String AS ev,
               countIf(ts >= toDateTime('{until}') - INTERVAL 1 DAY)  AS d1,
               countIf(ts >= toDateTime('{until}') - INTERVAL 7 DAY)  AS d7,
               countIf(ts >= toDateTime('{until}') - INTERVAL 30 DAY) AS d30
        FROM shop.history_events WHERE ts < '{until}'
        GROUP BY product_id, ev""")
    wide = df.pivot_table(index="product_id", columns="ev", values=["d1", "d7", "d30"], fill_value=0)
    wide.columns = [f"{e}_{w}" for w, e in wide.columns]
    wide = np.log1p(wide.reindex(PIDS).fillna(0).astype(float))
    return wide

pop_tr = popularity(TRAIN_START)
pop_te = popularity(CUT)[pop_tr.columns]                 # mismas columnas y orden
mu, sd = pop_tr.mean(), pop_tr.std().replace(0, 1)
def pop_matrix(w):                                       # [P, F] estandarizado, fila 0 = desconocido
    return np.vstack([np.zeros(w.shape[1]), ((w - mu) / sd).to_numpy()]).astype(np.float32)
POP_TR, POP_TE = pop_matrix(pop_tr), pop_matrix(pop_te)
print(pop_tr.shape[1], "features de popularidad:", list(pop_tr.columns))

# Señal escalar de popularidad para la mezcla final: compras (o lo más parecido) en 7 días
buy_col = next((c for c in pop_tr.columns if "purch" in c and c.endswith("d7")), None) \
          or next(c for c in pop_tr.columns if c.endswith("d7"))
print("popularidad para la mezcla:", buy_col)
def pop_scalar(w): return np.r_[-1e9, w[buy_col].to_numpy()].astype(np.float32)
POPS_TR, POPS_TE = pop_scalar(pop_tr), pop_scalar(pop_te)

9 features de popularidad: ['add_to_cart_d1', 'purchase_d1', 'view_d1', 'add_to_cart_d30', 'purchase_d30', 'view_d30', 'add_to_cart_d7', 'purchase_d7', 'view_d7']
popularidad para la mezcla: purchase_d7


## Datos de entrenamiento

Aleatorios dentro de la ventana y solo con clic (los demás no aportan positivos). `clicked` y `purchased` cuentan como positivos.

In [9]:
raw = sql(f"""
    SELECT request_id, ts, context, recent_views, shown, clicked, purchased
    FROM shop.history_sessions
    WHERE ts >= '{TRAIN_START}' AND ts < '{CUT}' AND clicked::String != ''
    ORDER BY rand() LIMIT 300000""")
raw["clicked"] = raw.clicked.replace("", None); raw["purchased"] = raw.purchased.replace("", None)
raw = raw[raw.clicked.notna()].reset_index(drop=True)
raw_val = raw[raw.ts.astype(str) >= VAL_START].reset_index(drop=True)
raw_tr  = raw[raw.ts.astype(str) <  VAL_START].reset_index(drop=True)
print(len(raw_tr), "train /", len(raw_val), "val (impresiones con clic)")

CTX_KEYS = ["device", "region", "page"]
CTX = {k: {v: i + 1 for i, v in enumerate(sorted({str((c or {}).get(k)) for c in raw_tr.context}))} for k in CTX_KEYS}

def to_unix(x): return pd.Timestamp(x).value / 1e9

def encode_rows(df):
    """Pasa peticiones (DataFrame con context, recent_views, ts) a tensores.
    Es la misma función en entrenamiento, validación, test y en vivo."""
    n = len(df)
    ctx = np.zeros((n, 3), np.int64)
    hist = np.zeros((n, MAX_HISTORY), np.int64)
    age = np.zeros((n, MAX_HISTORY), np.float32)
    for i, (c, views, ts) in enumerate(zip(df.context, df.recent_views, df.ts)):
        c = c if isinstance(c, dict) else {}
        ctx[i] = [CTX[k].get(str(c.get(k)), 0) for k in CTX_KEYS]
        views = [v for v in (views if isinstance(views, list) else []) if isinstance(v, dict) and v.get("product_id")]
        views = sorted(views, key=lambda v: str(v.get("ts")), reverse=True)[:MAX_HISTORY]
        t0 = to_unix(ts)
        for j, v in enumerate(views):
            hist[i, j] = PIDX.get(str(v["product_id"]), 0)
            age[i, j] = max(t0 - to_unix(v["ts"]), 0) / 3600 if v.get("ts") else 0
    return torch.from_numpy(ctx), torch.from_numpy(hist), torch.from_numpy(age)

def encode_slates(df):
    slate = np.zeros((len(df), 10), np.int64); pos = np.zeros((len(df), 10), bool)
    for i, (sh, c, b) in enumerate(zip(df.shown, df.clicked, df.purchased)):
        positives = {str(c), str(b)}
        for j, p in enumerate(list(sh)[:10]):
            slate[i, j] = PIDX.get(str(p), 0); pos[i, j] = str(p) in positives
    return torch.from_numpy(slate), torch.from_numpy(pos)

t = time.time()
TR = (*encode_rows(raw_tr), *encode_slates(raw_tr))
VA = (*encode_rows(raw_val), *encode_slates(raw_val))
print(f"encoding {time.time() - t:.1f}s")

14951 train / 2520 val (impresiones con clic)
encoding 0.2s


## Modelo

- **Torre de producto**: `[cat, subcat, marca, log precio, popularidad, ID] → MLP → vector normalizado`. Las features viven como *buffers* indexados por producto, así que la torre recibe solo índices. La popularidad es un buffer que se cambia entre train y test (`set_pop`) sin reentrenar, igual que en vivo cuando se actualice.
- **Torre de usuario**: media ponderada por recencia (vida media aprendible) de los productos vistos, más el último visto (intención inmediata), más el contexto. Un usuario sin historial queda con vector de historial 0 y solo contexto: cold start.

In [10]:
class ProductTower(nn.Module):
    def __init__(self, pop, id_dropout=0.3):
        super().__init__()
        for name, arr in [("cat", cat_arr), ("sub", sub_arr), ("brand", brand_arr)]:
            self.register_buffer(name, torch.tensor(arr, dtype=torch.long))
        self.register_buffer("price", torch.tensor(price_arr, dtype=torch.float32))
        self.register_buffer("pop", torch.tensor(pop))
        self.e_cat, self.e_sub, self.e_brand = nn.Embedding(n_cat, 8), nn.Embedding(n_sub, 16), nn.Embedding(n_brand, 16)
        self.e_id = nn.Embedding(P, 16); nn.init.zeros_(self.e_id.weight)   # ID nuevo = vector 0
        self.id_dropout = id_dropout
        d = 8 + 16 + 16 + 1 + pop.shape[1] + 16
        self.mlp = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Linear(128, EMB_DIM))
    def set_pop(self, pop): self.pop.copy_(torch.tensor(pop))
    def forward(self, idx):
        ids = idx
        if self.training and self.id_dropout > 0:
            ids = idx * (torch.rand(idx.shape, device=idx.device) > self.id_dropout)
        x = torch.cat([self.e_cat(self.cat[idx]), self.e_sub(self.sub[idx]), self.e_brand(self.brand[idx]),
                       self.price[idx].unsqueeze(-1), self.pop[idx], self.e_id(ids)], -1)
        return F.normalize(self.mlp(x), dim=-1)

class UserTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.e_ctx = nn.ModuleList([nn.Embedding(len(CTX[k]) + 1, 4) for k in CTX_KEYS])
        self.log_tau = nn.Parameter(torch.tensor(math.log(24.0)))           # vida media (horas), aprendida
        self.mlp = nn.Sequential(nn.Linear(2 * EMB_DIM + 12, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, EMB_DIM))
    def forward(self, ctx, hist_emb, age, mask):
        w = torch.exp(-math.log(2) * age / self.log_tau.exp()) * mask
        mean = (hist_emb * w.unsqueeze(-1)).sum(1) / w.sum(1, keepdim=True).clamp(min=1e-6)
        last = hist_emb[:, 0] * mask[:, :1]                                  # el más reciente
        c = torch.cat([e(ctx[:, i]) for i, e in enumerate(self.e_ctx)], -1)
        return F.normalize(self.mlp(torch.cat([mean, last, c], -1)), dim=-1)

class TwoTowers(nn.Module):
    def __init__(self, pop):
        super().__init__()
        self.prod, self.user = ProductTower(pop), UserTower()
        self.logit_scale = nn.Parameter(torch.tensor(math.log(10.0)))
    def user_emb(self, ctx, hist, age):
        mask = (hist > 0).float()
        return self.user(ctx, self.prod(hist), age, mask)
    def scale(self): return self.logit_scale.exp().clamp(max=50)

model = TwoTowers(POP_TR).to(DEV)
print(sum(p.numel() for p in model.parameters()), "parámetros")

126054 parámetros


## Entrenamiento

Por cada impresión, el softmax compite sobre **[los 10 mostrados] ∪ [256 aleatorios del catálogo, compartidos en el batch]**. Los mostrados enseñan las preferencias finas y los aleatorios enseñan a descartar el resto del catálogo. La pérdida es multi-positiva: `−log Σ_pos softmax`.

Como validación usamos **hit@10 sobre el catálogo completo** en los 2 días de validación, la misma métrica que reporta `evaluate()`, y paramos cuando deja de mejorar.

In [11]:
N_NEG, BS = 256, 512

def loss_fn(m, ctx, hist, age, slate, pos):
    u = m.user_emb(ctx, hist, age)                                    # [B, D]
    s_emb = m.prod(slate)                                             # [B, 10, D]
    neg = torch.randint(1, P, (N_NEG,), device=DEV)
    l_slate = (s_emb * u.unsqueeze(1)).sum(-1)                        # [B, 10]
    l_neg = u @ m.prod(neg).T                                         # [B, N_NEG]
    logits = m.scale() * torch.cat([l_slate.masked_fill(slate == 0, -1e4), l_neg], 1)
    pos_mask = torch.cat([pos, torch.zeros_like(l_neg, dtype=torch.bool)], 1)
    return (torch.logsumexp(logits, 1) - torch.logsumexp(logits.masked_fill(~pos_mask, -1e4), 1)).mean()

@torch.no_grad()
def all_scores(m, ctx, hist, age, bs=4096):
    """Coseno usuario-producto para todo el catálogo: [N, P] (en CPU)."""
    m.eval()
    items = m.prod(torch.arange(P, device=DEV))
    out = [ (m.user_emb(ctx[i:i+bs].to(DEV), hist[i:i+bs].to(DEV), age[i:i+bs].to(DEV)) @ items.T).cpu()
            for i in range(0, len(ctx), bs) ]
    return torch.cat(out)

def hit_at_10(scores, hist, clicked_idx, exclude_viewed=False):
    s = scores.clone(); s[:, 0] = -1e9
    if exclude_viewed: s.scatter_(1, hist, -1e9)
    top = s.topk(10, 1).indices
    return (top == clicked_idx.unsqueeze(1)).any(1).float().mean().item()

va_click = torch.tensor([PIDX.get(str(c), 0) for c in raw_val.clicked])
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
best_hit, best_state, patience = -1, None, 0
for epoch in range(30):
    model.train(); perm = torch.randperm(len(TR[0])); tot = 0
    for i in range(0, len(perm), BS):
        b = [x[perm[i:i+BS]].to(DEV) for x in TR]
        loss = loss_fn(model, *b); opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); tot += loss.item() * len(b[0])
    h = hit_at_10(all_scores(model, *VA[:3]), VA[1], va_click)
    print(f"epoch {epoch+1:2d}  loss {tot/len(perm):.4f}  val hit@10 {h:.4f}  tau {model.user.log_tau.exp().item():.1f}h")
    if h > best_hit: best_hit, best_state, patience = h, {k: v.clone() for k, v in model.state_dict().items()}, 0
    else:
        patience += 1
        if patience >= 3: break
model.load_state_dict(best_state)
print("mejor val hit@10:", round(best_hit, 4))

epoch  1  loss 2.9055  val hit@10 0.4980  tau 23.0h
epoch  2  loss 2.5032  val hit@10 0.6194  tau 21.5h
epoch  3  loss 2.3056  val hit@10 0.7155  tau 20.0h
epoch  4  loss 2.2138  val hit@10 0.7274  tau 19.2h
epoch  5  loss 2.2008  val hit@10 0.7234  tau 18.8h
epoch  6  loss 2.1623  val hit@10 0.7389  tau 18.6h
epoch  7  loss 2.1333  val hit@10 0.7091  tau 18.9h
epoch  8  loss 2.1015  val hit@10 0.7016  tau 19.4h
epoch  9  loss 2.0895  val hit@10 0.6980  tau 20.1h
mejor val hit@10: 0.7389


## Mezcla con popularidad (se ajusta en validación)

`final = cos(usuario, producto) + β · z(log compras 7d)`. Probamos varios β y si se excluyen o no los productos ya vistos. Todo se elige **en validación**; el test solo se mira una vez al final.

Con β = ∞ el ranking queda en popularidad pura. Si es lo que mejor funciona en validación, el two-tower no aporta y lo diremos así.

In [12]:
S_va = all_scores(model, *VA[:3])
pz = torch.tensor((POPS_TR - POPS_TR[1:].mean()) / POPS_TR[1:].std())
rows = []
for beta in [0, 0.05, 0.1, 0.2, 0.4, 0.8, 1.6, 1e3]:
    for ex in [False, True]:
        rows.append({"beta": beta, "exclude_viewed": ex,
                     "val_hit@10": hit_at_10(S_va + beta * pz, VA[1], va_click, ex)})
grid = pd.DataFrame(rows).sort_values("val_hit@10", ascending=False)
BETA, EXCL = grid.iloc[0].beta, bool(grid.iloc[0].exclude_viewed)
print("elegido: beta =", BETA, " exclude_viewed =", EXCL)
grid.head(8)

elegido: beta = 0.05  exclude_viewed = False


,beta,exclude_viewed,val_hit@10
2,0.05,False,0.753175
4,0.10,False,0.748413
0,0.00,False,0.738889
6,0.20,False,0.735714
8,0.40,False,0.686111
3,0.05,True,0.678968
5,0.10,True,0.675397
1,0.00,True,0.665873


## Evaluación en test (mismo `evaluate()` que el baseline)

Para test se cambia la popularidad a la calculada hasta `CUT`. Es lo que haría el servicio en vivo: refrescar la popularidad sin reentrenar.

In [13]:
model.prod.set_pop(POP_TE)
pz_te = torch.tensor((POPS_TE - POPS_TE[1:].mean()) / POPS_TE[1:].std())
t = time.time()
TE = encode_rows(test)
S_te = all_scores(model, *TE) + BETA * pz_te
S_te[:, 0] = -1e9
if EXCL: S_te.scatter_(1, TE[1], -1e9)
top = S_te.topk(10, 1).indices.numpy()
dt = time.time() - t
REC_TT2 = {rid: [PIDS[j - 1] for j in r] for rid, r in zip(test.request_id, top)}
print(f"{len(test)} peticiones en {dt:.1f}s ({1000*dt/len(test):.2f} ms/petición)")

def two_tower_v2(row): return REC_TT2[row.request_id]
def pop_only(row):                       # control: misma señal de popularidad sin el modelo
    return POP_ONLY
POP_ONLY = [PIDS[j - 1] for j in np.argsort(-POPS_TE)[:10]]

POLICIES = {"baseline": baseline, "coview": coview, "pop7d_only": pop_only, "two_tower_v2": two_tower_v2}
pd.DataFrame({n: evaluate(p, test) for n, p in POLICIES.items()}).T

100000 peticiones en 11.3s (0.11 ms/petición)


/tmp/ipykernel_592697/3756167819.py:11: RuntimeWarning: invalid value encountered in divide
  w = match / df.propensity.str[0].to_numpy()


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.386674,NaN,NaN,3971.0
coview,0.237624,NaN,NaN,5459.0
pop7d_only,0.386674,NaN,NaN,3971.0
two_tower_v2,0.756757,NaN,NaN,18326.0


## Cómo leer el resultado

- **hit@10** es la métrica con más señal (miles de clics). La comparación clave es `two_tower_v2` frente a `baseline` **y frente a `pop7d_only`**: si solo gana al baseline pero no a `pop7d_only`, la mejora viene de la popularidad reciente y no del modelo.
- **IPS/SNIPS slot 1**: mira siempre `slot1_matches`. Con menos de ~1.000 coincidencias el estimador es muy ruidoso.
- Si va bien, al pasarlo a vivo: `encode_rows` + `all_scores` sobre los ~200 candidatos de `get_candidates` (o sobre todo el catálogo, que cuesta poco) dentro de `score()`, y `set_pop()` cada pocos minutos con la popularidad de `shop.events`. Los productos nuevos entran con ID desconocido y se apoyan en categoría, marca, precio y popularidad, que es justo para lo que sirve el ID-dropout.

In [14]:
# 1) ¿Está el producto clicado en recent_views? ¿Hay vistas posteriores a la petición?
def views_of(r): return [v for v in (r.recent_views or []) if isinstance(v, dict)]
clk = test[test.clicked.notna()]
in_views = np.mean([r.clicked in {str(v["product_id"]) for v in views_of(r)} for r in clk.itertuples()])
future = np.mean([any(pd.Timestamp(v["ts"]) >= pd.Timestamp(r.ts) for v in views_of(r)) for r in test.itertuples()])
print(f"clic que ya estaba en recent_views: {in_views:.3f}")
print(f"peticiones con alguna vista con ts >= ts de la petición: {future:.3f}")

# 2) Política trivial: los 10 últimos vistos (sin repetir) y, si faltan, el baseline
def last_viewed(row):
    seen = list(dict.fromkeys(str(v["product_id"]) for v in sorted(views_of(row), key=lambda v: str(v["ts"]), reverse=True)))
    return (seen + [p for p in baseline(row) if p not in seen])[:10]

# 3) Evaluate tolerante a propensities vacías (de ahí los NaN)
def evaluate(policy, df):
    df = df[df.propensity.str[0].notna()]
    recs = [policy(r) for r in df.itertuples()]
    clicked, has = df.clicked.to_numpy(), df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has].mean()
    first = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first)])
    reward = (clicked == first).astype(float)
    w = match / df.propensity.str[0].astype(float).to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": np.nanmean(w * reward),
            "SNIPS_ctr_slot1": np.nansum(w * reward) / max(np.nansum(w), 1e-9), "slot1_matches": int(match.sum())}

POLICIES["last_viewed"] = last_viewed
pd.DataFrame({n: evaluate(p, test) for n, p in POLICIES.items()}).T

clic que ya estaba en recent_views: 0.101
peticiones con alguna vista con ts >= ts de la petición: 0.000


/tmp/ipykernel_592697/2727914458.py:23: RuntimeWarning: invalid value encountered in divide
  w = match / df.propensity.str[0].astype(float).to_numpy()
/tmp/ipykernel_592697/2727914458.py:23: RuntimeWarning: invalid value encountered in divide
  w = match / df.propensity.str[0].astype(float).to_numpy()
/tmp/ipykernel_592697/2727914458.py:23: RuntimeWarning: invalid value encountered in divide
  w = match / df.propensity.str[0].astype(float).to_numpy()
/tmp/ipykernel_592697/2727914458.py:23: RuntimeWarning: invalid value encountered in divide
  w = match / df.propensity.str[0].astype(float).to_numpy()
/tmp/ipykernel_592697/2727914458.py:23: RuntimeWarning: invalid value encountered in divide
  w = match / df.propensity.str[0].astype(float).to_numpy()


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.386674,0.006291,0.042559,3971.0
coview,0.237624,0.008602,0.011236,5459.0
pop7d_only,0.386674,0.006291,0.042559,3971.0
two_tower_v2,0.756757,0.020267,0.025064,18326.0
last_viewed,0.384801,0.008788,0.004528,1380.0
